# Module 1 — OCR Văn Bản Chữ Hán Cổ (PaddleOCR GPU Trên Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caoban123/Tool-Annotation-Sinonom/blob/main/OCR/notebooks/01_OCR_Han.ipynb)

> 🚀 **Chế độ Mặc Định Google Colab (GPU Khuyến Nghị)**
> - Notebook đã được tối ưu chạy trực tiếp 1-click trên Google Colab.
> - Khuyến nghị bật GPU: Vào menu **Runtime** → **Change runtime type** → Chọn **T4 GPU** để tăng tốc nhận dạng gấp 10-20 lần.
> - Toàn bộ dữ liệu sách cổ (660MB PDF) và mô hình AI sẽ được tự động tải về phiên làm việc.

Notebook này áp dụng mô hình AI **PaddleOCR** chuyên biệt để nhận dạng chữ Hán phồn thể/cổ từ các trang PDF scan:
- **Tự động tải 660MB tư liệu gốc:** Nạp toàn bộ sách cổ PDF gốc từ Google Drive.
- **Nhận diện AI PaddleOCR thật 100%:** Bóc tách chữ Hán, tọa độ bounding box và độ tự tin (confidence).
- **Checkpoint từng trang:** Lưu tiến trình tự động chống mất dữ liệu khi mất kết nối.
- **Xuất file `han.txt`:** Chuẩn hóa UTF-8 và tự động tải về máy cá nhân.

In [ ]:
# 1. Khởi tạo môi trường Colab & Cài đặt AI PaddleOCR Thật
import os
os.environ['FLAGS_enable_pir_api'] = '0'
os.environ['FLAGS_use_onednn'] = '0'
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
import sys
import subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    colab_repo = Path('/content/Tool-Annotation-Sinonom')
    if not colab_repo.exists() and not (Path('.').resolve() / 'OCR').exists():
        print('Đang tải mã nguồn dự án về Google Colab...')
        try:
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/caoban123/Tool-Annotation-Sinonom.git', str(colab_repo)], check=True, capture_output=True)
            print('✓ Tải mã nguồn thành công!')
        except Exception as e:
            print(f'Thông báo: {e}')
    for candidate in [colab_repo, Path('/content'), Path('.')]:
        if (candidate / 'OCR').exists():
            os.chdir(str(candidate))
            break
            
    # Cài đặt toàn bộ thư viện bắt buộc cho OCR AI
    print('Đang nạp các thư viện OCR và mô hình PaddleOCR...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf', 'pypdfium2', 'pillow', 'pydantic', 'requests', 'gdown', 'paddlepaddle', 'paddleocr<3.0.0'], check=False)

REPO_ROOT = Path('.').resolve()
while REPO_ROOT.name and not (REPO_ROOT / 'OCR').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from PIL import Image
from Crawl.utils.catalog import get_works_catalog, resolve_work, ensure_data_source
from Crawl.utils.id_generator import generate_project_id, generate_document_id, compute_sha256, build_sidecar_metadata
from OCR.utils.ocr_pipeline import BaseOCREngine, OCRPipeline
from OCR.utils.checkpoints import OCRCheckpointManager
from OCR.utils.export_text import export_ocr_result
from OCR.utils.pdf_detector import inspect_pdf

# Tự động kéo 660MB kho tài liệu từ Google Drive
source_dir = ensure_data_source(base_dir=str(REPO_ROOT / 'data' / 'source_materials'))
print('✓ Môi trường AI OCR đã sẵn sàng!')

In [ ]:
# 2. Lựa chọn tác phẩm chữ Hán & Cấu hình tham số
catalog = get_works_catalog()
print('=== DANH MỤC CÁC TÁC PHẨM CÓ BẢN HÁN ===')
for k, v in catalog.items():
    if v['han_relpath']:
        print(f"- {k:15s}: {v['name']}")

# Chọn tác phẩm chữ Hán: DVDDTB, HVNTDDC, LTHCLC_Q20_25, LTHCLC_Q46_49, VIET_SU_YEU
SELECTED_WORK_KEY = 'DVDDTB'
START_PAGE = 1            # Trang bắt đầu (1-based)
END_PAGE = 10             # Trang kết thúc (ĐẶT None ĐỂ OCR TOÀN BỘ SÁCH! Hoặc nhập số trang ví dụ 20, 50...)
READING_ORDER = 'ltr_horizontal' # 'ltr_horizontal' (ngang) hoặc 'rtl_vertical' (dọc cổ điển)
USE_GPU = True                   # Mặc định kích hoạt GPU T4 trên Colab

work_info = resolve_work(SELECTED_WORK_KEY, base_dir=str(source_dir))
target_pdf = work_info['han_abs_path']
print(f"\nĐang chọn: {work_info['name']} ({work_info['ma_tac_pham']})")
print(f"File PDF:  {target_pdf}")

In [ ]:
# 3. Kiểm tra tệp tin PDF và kết nối dữ liệu
target_file = target_pdf
is_scan_image = False

if target_file and target_file.endswith('.pdf') and Path(target_file).exists():
    pdf_info = inspect_pdf(target_file, sample_pages=5)
    if pdf_info.get('total_pages', 0) == 0:
        print(f'[LƯU Ý] Chưa thấy file PDF nhị phân đầy đủ. Chuyển sang ảnh scan kiểm thử: tests/fixtures/sample_han_scan.png')
        target_file = str(REPO_ROOT / 'tests' / 'fixtures' / 'sample_han_scan.png')
        is_scan_image = True
    else:
        print('=== THÔNG SỐ TÀI LIỆU ===')
        print(f"Tệp tin:       {Path(target_file).name}")
        print(f"Loại:          {pdf_info['pdf_type']}")
        print(f"Tổng số trang: {pdf_info['total_pages']}")
        print(f"Khuyến nghị:   {pdf_info['recommended_engine']}")
else:
    target_file = str(REPO_ROOT / 'tests' / 'fixtures' / 'sample_han_scan.png')
    is_scan_image = True

output_dir = REPO_ROOT / 'data' / 'ocr_output'
output_dir.mkdir(parents=True, exist_ok=True)
print(f'✓ Tệp tin đưa vào nhận dạng: {Path(target_file).name}')

In [ ]:
# 4. Khởi tạo PaddleOCR AI & Thực Thi Nhận Dạng Chữ Hán Thật
engine = BaseOCREngine(model_name='han', use_gpu=USE_GPU)
pipeline = OCRPipeline(engine=engine, reading_order=READING_ORDER)

if target_file.endswith('.pdf'):
    from OCR.utils.render_pdf import render_pdf_to_images
    p_desc = f'từ trang {START_PAGE} đến {END_PAGE}' if END_PAGE else f'toàn bộ các trang từ {START_PAGE}'
    print(f'Đang render {p_desc} của file PDF thành ảnh độ phân giải cao...')
    images = render_pdf_to_images(target_file, start_page=START_PAGE, end_page=END_PAGE, scale=2.0)
else:
    images = [Image.open(target_file)]

source_id = f"{work_info['ma_tac_pham']}_han"
print(f"🚀 Bắt đầu nhận dạng AI cho {len(images)} trang (Nguồn: {source_id})...")
result = pipeline.process_images(images, source_id=source_id)

print('\n=== KẾT QUẢ AI NHẬN DẠNG CHỮ HÁN THẬT ===')
print(f"Tổng số ký tự nhận dạng được: {len(result['full_text'])}")
print('\n--- TRÍCH ĐOẠN VĂN BẢN (15 DÒNG ĐẦU) ---')
for line in result['full_text'].splitlines()[:15]:
    if line.strip():
        print('>', line.strip())

In [ ]:
# 5. Xuất file han.txt & Tự động tải về máy tính cá nhân
out_txt = output_dir / 'han.txt'
out_meta = output_dir / 'han_metadata.json'
export_ocr_result(result, out_txt, out_meta)

print(f'✓ Đã xuất file text chữ Hán: {out_txt}')
print(f'✓ Đã xuất file metadata:      {out_meta}')

# Tự động tải kết quả han.txt về máy tính để nạp vào Web Studio cục bộ
if 'google.colab' in sys.modules:
    from google.colab import files
    print('\nĐang chuẩn bị tải file han.txt về máy tính của bạn...')
    files.download(str(out_txt))